# 11 — Phân loại toàn corpus bằng cascade ViSoBERT (Kaggle, GPU)

**Chiến lược downstream**

Theo amendment v2.2, chiến lược phân loại cho RQ2/RQ3 là cascade ViSoBERT: tầng 1 ACTIONABLE / NOT_ACTIONABLE với ngưỡng đóng băng
0,11, tầng 2 BUG / FEATURE / UX lấy lớp có xác suất cao nhất. Cấu hình đọc từ `03_configs/frozen/` (v2.1), không tinh chỉnh gì thêm.

**Train lại một seed**

Giai đoạn 10 không lưu mô hình, nên tầng 1 và tầng 2 được train lại với seed 42 — lần chạy chuẩn theo Q6 — trên Train 1.200 như ở
GĐ10. Trước khi ghi kết quả, dự đoán của mô hình mới được đối chiếu với dự đoán seed 42 đã lưu ở GĐ10.

**Phạm vi phân loại**

Toàn bộ corpus Shopee đã làm sạch (`shopee_clean_T0_2026-10-06.jsonl`, 56.050 review), dùng trường `text_clean` — cùng trường với
cột `text` của các file gold. Review thuộc Train, Validation, Hold-out và focused set vẫn nằm trong corpus vì gom cụm chạy trên toàn
bộ review actionable; mỗi review được gắn cột `split` để truy vết.

**Không dùng nhãn gold của tập khóa**

Corpus không có cột nhãn (notebook dừng nếu có); notebook cũng dừng nếu phát hiện file có nhãn của Hold-out, focused set hay Lazada
trong `/kaggle/input`.

In [ ]:
import os, subprocess, sys, json, gc, datetime
from kaggle_secrets import UserSecretsClient
REPO_URL = 'github.com/dnhphuong-05/kltn-vn-app-review-requirements.git'
REPO_DIR = '/kaggle/working/repo'
token = UserSecretsClient().get_secret('GITHUB_TOKEN')
if not os.path.exists(REPO_DIR):
    r = subprocess.run(['git', 'clone', '-q', f'https://{token}@{REPO_URL}', REPO_DIR], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('git clone thất bại: ' + r.stderr.replace(token, '***'))
subprocess.run(['git', '-C', REPO_DIR, 'remote', 'set-url', 'origin', f'https://{REPO_URL}'], check=True)
del token
sys.path.append(os.path.join(REPO_DIR, '02_src'))
import numpy as np, pandas as pd, torch, transformers, sentencepiece
from src.utils.seeds import set_seed
set_seed(42)
COMMIT = subprocess.run(['git', '-C', REPO_DIR, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
print('Ngày chạy:', datetime.date.today().isoformat(), '| commit:', COMMIT)
print('transformers', transformers.__version__, '| sentencepiece', sentencepiece.__version__, '| torch', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có')

## Cấu hình đóng băng

**Nguồn cấu hình**

`load_frozen` kiểm tra header FROZEN v2.1 của các file trong `03_configs/frozen/`. Notebook dùng cấu hình ViSoBERT của tầng 1 (kèm
ngưỡng) và tầng 2, với seed 42.

In [ ]:
from src.classification.final_eval import load_frozen, LOCKED
FZ = load_frozen(REPO_DIR)
SEED = 42
assert SEED in FZ['tier1']['seeds'] and SEED in FZ['tier2']['seeds']
CFG1, CFG2 = FZ['tier1']['visobert']['config'], FZ['tier2']['visobert']['config']
THR = FZ['tier1']['visobert']['threshold']
print('Tầng 1:', {k: CFG1[k] for k in ('lr', 'epochs', 'max_len')}, '| ngưỡng', THR)
print('Tầng 2:', {k: CFG2[k] for k in ('lr', 'epochs', 'max_len')})

## Dữ liệu

**Các file cần trên Kaggle**

`shopee_train.csv`, `shopee_validation.csv` (có nhãn, để train và đối chiếu) và `shopee_clean_T0_2026-10-06.jsonl` (corpus đã ẩn
danh hóa, không có nhãn), tìm bằng `os.walk` trong `/kaggle/input`.

**Gắn tập cho từng review**

Cột `split` lấy từ `01_data/manifests/splits_manifest_T0_2026-10-06.json` trong repo (chỉ chứa review_id): train, validation,
holdout, focused, hoặc none.

**Kiểm tra nhất quán**

Mọi review của Train và Validation phải có trong corpus và có cùng văn bản với file gold.

In [ ]:
from src.classification.full_inference import (CORPUS_FILE, SPLITS_MANIFEST, load_corpus, split_tags, cascade_table,
                                               class_summary, expected_positive_rate, agreement, actionable_export,
                                               PILOT_PREVALENCE)

def find_file(name, root='/kaggle/input'):
    hits = [os.path.join(d, name) for d, _, fs in os.walk(root) if name in fs]
    if not hits:
        listing = [os.path.join(d, f) for d, _, fs in os.walk(root) for f in fs]
        raise FileNotFoundError(f'Không tìm thấy {name} trong {root}. Các file hiện có:\n' + ('\n'.join(listing) or '(trống)'))
    return hits[0]

labeled_locked = [os.path.join(d, f) for d, _, fs in os.walk('/kaggle/input') for f in fs if f in LOCKED.values()]
assert not labeled_locked, f'Phát hiện file CÓ NHÃN của tập khóa trên Kaggle: {labeled_locked}'
train_df = pd.read_csv(find_file('shopee_train.csv'), encoding='utf-8-sig')
val_df = pd.read_csv(find_file('shopee_validation.csv'), encoding='utf-8-sig')
corpus = load_corpus(find_file(CORPUS_FILE))
corpus['split'] = split_tags(os.path.join(REPO_DIR, '01_data', 'manifests', SPLITS_MANIFEST), corpus['review_id']).values

text_of = corpus.set_index('review_id')['text']
for name, df in (('Train', train_df), ('Validation', val_df)):
    assert df['review_id'].isin(text_of.index).all(), f'{name}: có review không nằm trong corpus'
    assert (text_of.loc[df['review_id']].values == df['text'].values).all(), f'{name}: văn bản khác corpus'
print('Corpus:', len(corpus), 'review | Train', len(train_df), '| Validation', len(val_df))
print('Số review theo tập:', corpus['split'].value_counts().to_dict())
assert len(corpus) == 56050

## Kiểm tra tokenizer ViSoBERT

**Hai phép kiểm tra như GĐ10**

`self_check` so cách tách với `spm.encode` gốc; `embedding_alignment` che từng token để đầu MLM đoán lại, dừng nếu dưới 0,3.

In [ ]:
from src.classification.strategy_visobert.tokenizer import VisobertTokenizer, self_check, embedding_alignment
VIS_NAME = CFG1['model_name']
tok = VisobertTokenizer(VIS_NAME)
self_check(tok)
align = embedding_alignment(tok, VIS_NAME, train_df['text'].astype(str).sample(60, random_state=42))
print(f"MLM đoán lại token bị che: {align['mlm_top1']:.1%} trên {align['n_tokens']} token")
assert align['mlm_top1'] > 0.3, 'id token không khớp embedding của ViSoBERT'
del tok; gc.collect(); torch.cuda.empty_cache()

## Train lại tầng 1 và tầng 2 (seed 42) và phân loại corpus

**Dữ liệu train**

Tầng 1: Train 1.200 (gồm ambiguous). Tầng 2: Train actionable, bỏ ambiguous — giống GĐ10.

**Dự đoán**

Mỗi tầng dự đoán Validation (để đối chiếu) và toàn bộ corpus. Tầng 2 dự đoán mọi review; nhãn tầng 2 chỉ được dùng cho review mà tầng 1
gọi ACTIONABLE. Thời gian dự đoán được đo trên corpus.

In [ ]:
from src.classification.rq1 import VisobertRunner, run_multi
from src.classification.tier1_actionable.mapping import to_tier1, TIER1_LABELS
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS
SETS = {'validation': val_df, 'corpus': corpus}
RUNS = {}
for tier, cfg, tr, names in (('tier1', CFG1, to_tier1(train_df), TIER1_LABELS),
                             ('tier2', CFG2, to_tier2(train_df, drop_ambiguous=True), TIER2_LABELS)):
    runner = VisobertRunner(cfg)
    assert runner.config(None) == cfg, f'{tier}: cấu hình hiệu lực khác file frozen'
    RUNS[tier] = run_multi(runner, tr, SETS, len(names), names, SEED, timing_set='corpus')
    r = RUNS[tier]
    print(f"{tier}: train {r['train_seconds']:.0f}s | dự đoán corpus {r['infer_seconds_per_1000']:.2f}s/1000 | "
          f"GPU {r['peak_gpu_gb']} GB | n_train {r['n_train']}")

## Đối chiếu với seed 42 của GĐ10

**So sánh**

Dự đoán của mô hình mới được so với dự đoán seed 42 đã lưu ở GĐ10 (`gd10_pred_validation.csv`, `gd10_pred_holdout.csv`,
`gd10_pred_focused.csv` trong repo): tỷ lệ nhãn khớp và chênh xác suất lớn nhất. Dự đoán Hold-out và focused set được lấy từ các
dòng tương ứng của corpus; chỉ so dự đoán với dự đoán, không dùng nhãn.

**Điều kiện ghi kết quả**

Trên Validation, Macro-F1 tầng 1 (ở ngưỡng 0,11) và Macro-F1 tầng 2 (gold-actionable) lệch không quá 2 điểm so với seed 42 của GĐ10,
và recall ACTIONABLE ≥ 0,80. Không đạt thì notebook dừng và không ghi kết quả.

In [ ]:
from src.classification import evaluate as E
from src.classification.tier1_actionable.threshold import metrics_at
from src.classification.tier1_actionable.mapping import to_tier1
from src.classification.tier2_bfu.mapping import to_tier2, TIER2_LABELS
RES = os.path.join(REPO_DIR, '05_experiments', 'rq1_classification')
P1, P2 = ['p_NOT_ACTIONABLE', 'p_ACTIONABLE'], ['p_BUG', 'p_FEATURE', 'p_UX']
pos = {rid: i for i, rid in enumerate(corpus['review_id'])}

def ref_probs(set_name, tier, ids):
    d = pd.read_csv(os.path.join(RES, f'gd10_pred_{set_name}.csv'))
    d = d[(d['tier'] == tier) & (d['model'] == 'visobert') & (d['seed'] == SEED)].set_index('review_id')
    return d.loc[list(ids), P1 if tier == 'tier1' else P2].to_numpy()

rows = []
for set_name in ('validation', 'holdout', 'focused'):
    ids = pd.read_csv(os.path.join(RES, f'gd10_pred_{set_name}.csv'), usecols=['review_id'])['review_id'].unique()
    for tier in ('tier1', 'tier2'):
        new = (RUNS[tier]['probs']['validation'] if set_name == 'validation'
               else RUNS[tier]['probs']['corpus'][[pos[r] for r in ids]])
        if set_name == 'validation':
            new = pd.DataFrame(new, index=val_df['review_id']).loc[list(ids)].to_numpy()
        a = agreement(new, ref_probs(set_name, tier, ids), THR if tier == 'tier1' else None)
        rows.append({'tập': set_name, 'tầng': tier, **a})
REPRO = pd.DataFrame(rows)
print(REPRO.round(4).to_string(index=False))

y1 = to_tier1(val_df)['y'].to_numpy()
v2 = to_tier2(val_df, drop_ambiguous=False)
mask2 = val_df['review_id'].isin(v2['review_id']).to_numpy()
ref1, ref2 = ref_probs('validation', 'tier1', val_df['review_id']), ref_probs('validation', 'tier2', val_df['review_id'])
new1, new2 = RUNS['tier1']['probs']['validation'], RUNS['tier2']['probs']['validation']
m_new, m_ref = metrics_at(y1, new1[:, 1], THR), metrics_at(y1, ref1[:, 1], THR)
t2_new = E.metrics(v2['y'].to_numpy(), new2[mask2].argmax(1), TIER2_LABELS)['macro_f1']
t2_ref = E.metrics(v2['y'].to_numpy(), ref2[mask2].argmax(1), TIER2_LABELS)['macro_f1']
CHECK = pd.DataFrame([
    {'chỉ số (Validation)': 'Macro-F1 tầng 1', 'GĐ10 seed 42': m_ref['macro_f1'], 'GĐ11': m_new['macro_f1']},
    {'chỉ số (Validation)': 'Recall ACTIONABLE', 'GĐ10 seed 42': m_ref['recall_actionable'], 'GĐ11': m_new['recall_actionable']},
    {'chỉ số (Validation)': 'Macro-F1 tầng 2', 'GĐ10 seed 42': t2_ref, 'GĐ11': t2_new}])
CHECK['chênh (điểm)'] = (CHECK['GĐ11'] - CHECK['GĐ10 seed 42']) * 100
PASS = bool(abs(CHECK.loc[0, 'chênh (điểm)']) <= 2 and abs(CHECK.loc[2, 'chênh (điểm)']) <= 2
            and m_new['recall_actionable'] >= FZ['tier1']['threshold_rule']['min_recall'])
print(); print(CHECK.round(4).to_string(index=False))
print('\nĐỐI CHIẾU:', 'ĐẠT — ghi kết quả' if PASS else 'KHÔNG ĐẠT — DỪNG')

## Kết quả trên toàn corpus

**Bảng phân loại**

Mỗi review có xác suất và nhãn tầng 1, xác suất tầng 2, nhãn tầng 2 (chỉ khi ACTIONABLE) và nhãn bốn lớp của cascade.

**Tỷ lệ actionable**

Tỷ lệ review được gọi ACTIONABLE trên toàn corpus được đặt cạnh prevalence của pilot (20,5%, CI 95% 16,8–24,7%, n = 400; amendment
v1.7) và tỷ lệ kỳ vọng của bộ phân loại = prevalence × recall / precision, với recall và precision tầng 1 của ViSoBERT trên Hold-out
(GĐ10, trung bình ba seed).

**File xuất**

`gd11_corpus_predictions.csv` (không có văn bản), `actionable_reviews.csv` (review_id, text, tier2_label, review_date, app_version,
split — đầu vào của GĐ12, có văn bản nên không commit), bảng tóm tắt, bảng đối chiếu và thông tin lần chạy.

In [ ]:
from src.utils.kaggle_paths import working_dir
from src.classification.final_eval import parse_pm
if not PASS:
    raise RuntimeError('Dự đoán của mô hình train lại lệch GĐ10 quá ngưỡng — không ghi kết quả.')
TABLE = cascade_table(corpus['review_id'], corpus['split'], RUNS['tier1']['probs']['corpus'], RUNS['tier2']['probs']['corpus'], THR)
SUMMARY = class_summary(TABLE)
print(SUMMARY.assign(**{'tỷ lệ ACTIONABLE': SUMMARY['tỷ lệ ACTIONABLE'].round(4)}).to_string(index=False))

h = pd.read_csv(os.path.join(RES, 'gd10_tier1_holdout.csv')).set_index('chiến lược').loc['ViSoBERT']
rec, prec = parse_pm(h['Recall ACT'])[0] / 100, parse_pm(h['Precision ACT'])[0] / 100
exp_rate = expected_positive_rate(PILOT_PREVALENCE['actionable'], rec, prec)
rate = SUMMARY.loc[0, 'tỷ lệ ACTIONABLE']
print(f"\nTỷ lệ ACTIONABLE trên toàn corpus: {rate:.1%} ({SUMMARY.loc[0, 'ACTIONABLE']:,} / {len(TABLE):,})")
print(f"Prevalence pilot: {PILOT_PREVALENCE['actionable']:.1%} (CI 95% {PILOT_PREVALENCE['ci95'][0]:.1%}–{PILOT_PREVALENCE['ci95'][1]:.1%})")
print(f'Tỷ lệ kỳ vọng của bộ phân loại (prevalence × recall {rec:.3f} / precision {prec:.3f}): {exp_rate:.1%}')

ACT = actionable_export(TABLE, corpus)
print('\nActionable theo nhãn tầng 2:', ACT['tier2_label'].value_counts().to_dict())
print('Actionable theo tập:', ACT['split'].value_counts().to_dict())

OUT = working_dir('rq2_gd11')
TABLE.to_csv(os.path.join(OUT, 'gd11_corpus_predictions.csv'), index=False)
ACT.to_csv(os.path.join(OUT, 'actionable_reviews.csv'), index=False, encoding='utf-8-sig')
SUMMARY.to_csv(os.path.join(OUT, 'gd11_class_summary.csv'), index=False)
REPRO.to_csv(os.path.join(OUT, 'gd11_repro_check.csv'), index=False)
CHECK.to_csv(os.path.join(OUT, 'gd11_validation_check.csv'), index=False)
json.dump({'run_date': datetime.date.today().isoformat(), 'commit': COMMIT, 'seed': SEED, 'threshold': THR,
           'n_corpus': len(TABLE), 'n_actionable': int(len(ACT)), 'actionable_rate': rate,
           'expected_rate': exp_rate, 'pilot_prevalence': PILOT_PREVALENCE['actionable'],
           'train_seconds': {t: RUNS[t]['train_seconds'] for t in RUNS},
           'infer_seconds_per_1000_corpus': {t: RUNS[t]['infer_seconds_per_1000'] for t in RUNS},
           'peak_gpu_gb': {t: RUNS[t]['peak_gpu_gb'] for t in RUNS},
           'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu',
           'transformers': transformers.__version__, 'sentencepiece': sentencepiece.__version__, 'torch': torch.__version__},
          open(os.path.join(OUT, 'gd11_run_info.json'), 'w'), indent=1)
print('\nĐã lưu:', sorted(os.listdir(OUT)))

## Ghi chú

**Bước tiếp theo**

Tải thư mục `rq2_gd11/` về `05_experiments/rq2_topics/`, rồi chạy notebook 12 trên máy local. Mô hình không được lưu: toàn bộ dự đoán
cần cho các giai đoạn sau nằm trong `gd11_corpus_predictions.csv`.